In [1]:
import pandas as pd
import requests
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

## Add the values of the LEZ areas

In [2]:
df_values = pd.read_csv(r"..\main\mean_monthlyvalues.csv")
df_LEZ_area = pd.read_csv(r"..\main\Visualisation\LEZ_area.csv")

In [3]:
df_values.head()

,Year_Month,Province,Average NO2 Value,Average PM2.5 Value,Average PM10 Value
0,1990-01,Drenthe,1.324917,NaN,NaN
1,1990-01,Flevoland,1.764091,NaN,NaN
2,1990-01,Friesland,1.215124,NaN,NaN
3,1990-01,Gelderland,2.422852,NaN,NaN
4,1990-01,Groningen,1.215124,NaN,NaN


In [4]:
df_LEZ_area.head()

,Unnamed: 0,zone_name,start_date,end_date,area_km2,province
0,0,LEZ 's-Hertogenbosch,2020-01-01T00:00:00Z,2025-03-01T00:00:00Z,2.856116,North-Brabant
1,1,LEZ Delft,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,1.374783,Zuid-Holland
2,2,LEZ Haarlem,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,11.509328,Noord-Holland
3,3,LEZ Apeldoorn,2025-01-01T00:00:00Z,2026-12-31T23:00:00Z,1.929479,Gelderland
4,4,LEZ Den Haag,2020-01-01T00:00:00Z,2999-01-01T00:00:00Z,12.110339,Zuid-Holland


In [5]:
# Standardize the date information
df_values['Year_Month'] = pd.to_datetime(df_values['Year_Month'], format='%Y-%m')

# Convert start_date and end_date to datetime
df_LEZ_area['start_date'] = pd.to_datetime(df_LEZ_area['start_date'], errors='coerce')
df_LEZ_area['end_date'] = pd.to_datetime(df_LEZ_area['end_date'], errors='coerce')

# Normalize to month start to match Year_Month convention
df_LEZ_area['start_date'] = df_LEZ_area['start_date'].dt.to_period('M').dt.to_timestamp()
df_LEZ_area['end_date'] = df_LEZ_area['end_date'].dt.to_period('M').dt.to_timestamp()

df_LEZ_area['end_date'] = df_LEZ_area['end_date'].fillna(pd.Timestamp('2100-12-01'))

C:\Users\Utente\AppData\Local\Temp\ipykernel_27932\3364337627.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_LEZ_area['start_date'] = df_LEZ_area['start_date'].dt.to_period('M').dt.to_timestamp()
C:\Users\Utente\AppData\Local\Temp\ipykernel_27932\3364337627.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_LEZ_area['end_date'] = df_LEZ_area['end_date'].dt.to_period('M').dt.to_timestamp()


In [6]:
df_LEZ_area.head()

,Unnamed: 0,zone_name,start_date,end_date,area_km2,province
0,0,LEZ 's-Hertogenbosch,2020-01-01,2025-03-01,2.856116,North-Brabant
1,1,LEZ Delft,2020-01-01,2100-12-01,1.374783,Zuid-Holland
2,2,LEZ Haarlem,2020-01-01,2100-12-01,11.509328,Noord-Holland
3,3,LEZ Apeldoorn,2025-01-01,2026-12-01,1.929479,Gelderland
4,4,LEZ Den Haag,2020-01-01,2100-12-01,12.110339,Zuid-Holland


In [7]:
print(df_LEZ_area)

    Unnamed: 0                 zone_name start_date   end_date    area_km2  \
0            0      LEZ 's-Hertogenbosch 2020-01-01 2025-03-01    2.856116   
1            1                 LEZ Delft 2020-01-01 2100-12-01    1.374783   
2            2               LEZ Haarlem 2020-01-01 2100-12-01   11.509328   
3            3             LEZ Apeldoorn 2025-01-01 2026-12-01    1.929479   
4            4              LEZ Den Haag 2020-01-01 2100-12-01   12.110339   
5            5                 LEZ Breda 2020-01-01 2100-12-01    2.863319   
6            6               LEZ Tilburg 2020-01-01 2025-01-01   14.858716   
7            7               LEZ Utrecht 2020-01-01 2100-12-01    5.308662   
8            8                LEZ Leiden 2020-01-01 2100-12-01    3.085373   
9            9              LEZ Rijswijk 2020-01-01 2100-12-01    2.463771   
10          10         Milieuzone Arnhem 2020-12-01 2100-12-01    1.137788   
11          11             LEZ Amsterdam 2020-11-01 2025-01-01  

In [8]:
print(df_LEZ_area['area_km2'].skew())

2.6056328717405917


In [9]:
df_LEZ_area['area_km2_log'] = np.log(df_LEZ_area['area_km2'])
print(df_LEZ_area['area_km2_log'])
print(df_LEZ_area['area_km2_log'].skew())

0     1.049463
1     0.318296
2     2.443158
3     0.657250
4     2.494060
5     1.051982
6     2.698587
7     1.669340
8     1.126673
9     0.901693
10    0.129086
11    4.633945
12   -3.440825
13    4.046084
Name: area_km2_log, dtype: float64
-0.76735308313608


In [10]:
# Standardize values for LEZ area
from sklearn.preprocessing import StandardScaler

standardization = df_LEZ_area[['area_km2_log']]

standardize = StandardScaler().fit_transform(standardization)

df_LEZ_area['area_km2_st'] = pd.DataFrame(standardize, columns=standardization.columns)

print(df_LEZ_area)

    Unnamed: 0                 zone_name start_date   end_date    area_km2  \
0            0      LEZ 's-Hertogenbosch 2020-01-01 2025-03-01    2.856116   
1            1                 LEZ Delft 2020-01-01 2100-12-01    1.374783   
2            2               LEZ Haarlem 2020-01-01 2100-12-01   11.509328   
3            3             LEZ Apeldoorn 2025-01-01 2026-12-01    1.929479   
4            4              LEZ Den Haag 2020-01-01 2100-12-01   12.110339   
5            5                 LEZ Breda 2020-01-01 2100-12-01    2.863319   
6            6               LEZ Tilburg 2020-01-01 2025-01-01   14.858716   
7            7               LEZ Utrecht 2020-01-01 2100-12-01    5.308662   
8            8                LEZ Leiden 2020-01-01 2100-12-01    3.085373   
9            9              LEZ Rijswijk 2020-01-01 2100-12-01    2.463771   
10          10         Milieuzone Arnhem 2020-12-01 2100-12-01    1.137788   
11          11             LEZ Amsterdam 2020-11-01 2025-01-01  

In [11]:
# Define the function that add the LEZ area
def get_area(row):
    province = row['Province']
    date = row['Year_Month']
    # Filter by province
    matches = df_LEZ_area[
        (df_LEZ_area['province'] == province) &
        (df_LEZ_area['start_date'] <= date) &
        (df_LEZ_area['end_date'] >= date)
    ]

    # Sum all matching areas (if multiple)
    area_sum = matches['area_km2_st'].sum() if not matches.empty else 0

    # Add to existing value if present, else just area_sum
    current_val = row['area_km2_st'] if pd.notna(row.get('area_km2_st', None)) else 0
    return current_val + area_sum

if 'area_km2_st' not in df_values.columns:
    df_values['area_km2_st'] = 0

df_values['area_km2_st'] = df_values.apply(get_area, axis=1)

In [12]:
display(df_values)

,Year_Month,Province,Average NO2 Value,Average PM2.5 Value,Average PM10 Value,area_km2_st
0,1990-01-01,Drenthe,1.324917,NaN,NaN,0.000000
1,1990-01-01,Flevoland,1.764091,NaN,NaN,0.000000
2,1990-01-01,Friesland,1.215124,NaN,NaN,0.000000
3,1990-01-01,Gelderland,2.422852,NaN,NaN,0.000000
4,1990-01-01,Groningen,1.215124,NaN,NaN,0.000000
...,...,...,...,...,...,...
1435,2024-12-01,Noord-Holland,0.336776,0.122001,-0.208369,2.276287
1436,2024-12-01,Overijssel,-0.870952,-0.956227,-1.362410,0.000000
1437,2024-12-01,Utrecht,-0.102398,0.391558,-0.669985,0.137367
1438,2024-12-01,Zeeland,-0.761158,NaN,-0.208369,0.000000


## Add the values for vehicles 

In [13]:
{
  "odata.metadata":"https://opendata.cbs.nl/ODataApi/OData/85235NED/$metadata","value":[
    {
      "name":"TableInfos","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/TableInfos"
    },{
      "name":"UntypedDataSet","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/UntypedDataSet"
    },{
      "name":"TypedDataSet","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/TypedDataSet"
    },{
      "name":"DataProperties","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/DataProperties"
    },{
      "name":"CategoryGroups","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/CategoryGroups"
    },{
      "name":"RegioS","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/RegioS"
    },{
      "name":"Perioden","url":"https://opendata.cbs.nl/ODataApi/odata/85235NED/Perioden"
    }
  ]
}

{'odata.metadata': 'https://opendata.cbs.nl/ODataApi/OData/85235NED/$metadata',
 'value': [{'name': 'TableInfos',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/TableInfos'},
  {'name': 'UntypedDataSet',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/UntypedDataSet'},
  {'name': 'TypedDataSet',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/TypedDataSet'},
  {'name': 'DataProperties',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/DataProperties'},
  {'name': 'CategoryGroups',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/CategoryGroups'},
  {'name': 'RegioS',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/RegioS'},
  {'name': 'Perioden',
   'url': 'https://opendata.cbs.nl/ODataApi/odata/85235NED/Perioden'}]}

In [14]:
url = "https://opendata.cbs.nl/ODataApi/odata/85235NED/TypedDataSet"
response= requests.get(url)
users= response.json()
print(users)

{'odata.metadata': 'https://opendata.cbs.nl/ODataApi/OData/85235NED/$metadata#Cbs.OData.WebAPI.TypedDataSet', 'value': [{'ID': 0, 'RegioS': 'NL01  ', 'Perioden': '2019JJ00', 'TotaalWegvoertuigen_1': 11372065, 'TotaalMotorvoertuigen_2': 10199679, 'TotaalPersonenautoS_3': 8442982, 'PersonenautoSRelatief_4': 489, 'InBezitNatuurlijkePersonen_5': 7452085, 'InBezitNatuurlijkePersonenRelatief_6': 431, 'TotaalBedrijfsvoertuigen_7': 2283037, 'TotaalBedrijfsmotorvoertuigen_8': 1110651, 'TotaalAanhangwagensEnOpleggers_9': 1172386, 'TotaalBedrijfsmotorvoertuigen_10': 1110651, 'Bestelauto_11': 903005, 'VrachtautoExclTrekkerVoorOplegger_12': 61652, 'TrekkerVoorOplegger_13': 78788, 'SpeciaalVoertuig_14': 57693, 'Bus_15': 9513, 'TotaalAanhangwagensEnOpleggers_16': 1172386, 'Aanhangwagen_17': 1008412, 'Oplegger_18': 163974, 'TotaalMotorfietsen_19': 646046, 'MotorfietsenRelatief_20': 37}, {'ID': 1, 'RegioS': 'NL01  ', 'Perioden': '2020JJ00', 'TotaalWegvoertuigen_1': 11567203, 'TotaalMotorvoertuigen_2': 

In [15]:
#this will print the typed dataset in json format, so name value pairs
url = "https://opendata.cbs.nl/ODataApi/odata/85235NED/TypedDataSet"
posts = requests.get(url).json()

print (posts)
print("test")

{'odata.metadata': 'https://opendata.cbs.nl/ODataApi/OData/85235NED/$metadata#Cbs.OData.WebAPI.TypedDataSet', 'value': [{'ID': 0, 'RegioS': 'NL01  ', 'Perioden': '2019JJ00', 'TotaalWegvoertuigen_1': 11372065, 'TotaalMotorvoertuigen_2': 10199679, 'TotaalPersonenautoS_3': 8442982, 'PersonenautoSRelatief_4': 489, 'InBezitNatuurlijkePersonen_5': 7452085, 'InBezitNatuurlijkePersonenRelatief_6': 431, 'TotaalBedrijfsvoertuigen_7': 2283037, 'TotaalBedrijfsmotorvoertuigen_8': 1110651, 'TotaalAanhangwagensEnOpleggers_9': 1172386, 'TotaalBedrijfsmotorvoertuigen_10': 1110651, 'Bestelauto_11': 903005, 'VrachtautoExclTrekkerVoorOplegger_12': 61652, 'TrekkerVoorOplegger_13': 78788, 'SpeciaalVoertuig_14': 57693, 'Bus_15': 9513, 'TotaalAanhangwagensEnOpleggers_16': 1172386, 'Aanhangwagen_17': 1008412, 'Oplegger_18': 163974, 'TotaalMotorfietsen_19': 646046, 'MotorfietsenRelatief_20': 37}, {'ID': 1, 'RegioS': 'NL01  ', 'Perioden': '2020JJ00', 'TotaalWegvoertuigen_1': 11567203, 'TotaalMotorvoertuigen_2': 

In [16]:
# select columns by first selecting the "value" key from json format
# then specify the specific columns
dfposts = pd.DataFrame(posts["value"])[['Aanhangwagen_17',
            'Bestelauto_11',
            'Bus_15',
            'ID',
            'InBezitNatuurlijkePersonenRelatief_6',
            'InBezitNatuurlijkePersonen_5',
            'MotorfietsenRelatief_20',
            'Oplegger_18',
            'Perioden',
            'PersonenautoSRelatief_4',
            'RegioS',
            'SpeciaalVoertuig_14',
            'TotaalAanhangwagensEnOpleggers_16',
            'TotaalAanhangwagensEnOpleggers_9',
            'TotaalBedrijfsmotorvoertuigen_10',
            'TotaalBedrijfsmotorvoertuigen_8',
            'TotaalBedrijfsvoertuigen_7',
            'TotaalMotorfietsen_19',
            'TotaalMotorvoertuigen_2',
            'TotaalPersonenautoS_3',
            'TotaalWegvoertuigen_1',
            'TrekkerVoorOplegger_13',
            'VrachtautoExclTrekkerVoorOplegger_12']]
dfposts.head()

,Aanhangwagen_17,Bestelauto_11,Bus_15,ID,InBezitNatuurlijkePersonenRelatief_6,InBezitNatuurlijkePersonen_5,MotorfietsenRelatief_20,Oplegger_18,Perioden,PersonenautoSRelatief_4,...,TotaalAanhangwagensEnOpleggers_9,TotaalBedrijfsmotorvoertuigen_10,TotaalBedrijfsmotorvoertuigen_8,TotaalBedrijfsvoertuigen_7,TotaalMotorfietsen_19,TotaalMotorvoertuigen_2,TotaalPersonenautoS_3,TotaalWegvoertuigen_1,TrekkerVoorOplegger_13,VrachtautoExclTrekkerVoorOplegger_12
0,1008412,903005,9513,0,431,7452085,37,163974,2019JJ00,489,...,1172386,1110651,1110651,2283037,646046,10199679,8442982,11372065,78788,61652
1,1023980,927251,9699,1,434,7548770,38,168502,2020JJ00,493,...,1192482,1135943,1135943,2328425,654387,10374721,8584391,11567203,80118,61746
2,1029636,945433,9050,2,440,7683236,38,170014,2021JJ00,497,...,1199650,1151740,1151740,2351390,666597,10504756,8686419,11704406,79932,60831
3,1051486,974792,8532,3,444,7803950,39,177294,2022JJ00,502,...,1228780,1182857,1182857,2411637,677787,10688353,8827709,11917133,82436,60867
4,1069827,989841,8756,4,440,7842473,39,185262,2023JJ00,501,...,1255089,1201061,1201061,2456150,690724,10808892,8917107,12063981,85679,60811


In [17]:
columns_vehicledb = ['Aanhangwagen_17',
            'Bestelauto_11',
            'Bus_15',
            'InBezitNatuurlijkePersonenRelatief_6',
            'InBezitNatuurlijkePersonen_5',
            'MotorfietsenRelatief_20',
            'Oplegger_18',
            'PersonenautoSRelatief_4',
            'SpeciaalVoertuig_14',
            'TotaalAanhangwagensEnOpleggers_16',
            'TotaalAanhangwagensEnOpleggers_9',
            'TotaalBedrijfsmotorvoertuigen_10',
            'TotaalBedrijfsmotorvoertuigen_8',
            'TotaalBedrijfsvoertuigen_7',
            'TotaalMotorfietsen_19',
            'TotaalMotorvoertuigen_2',
            'TotaalPersonenautoS_3',
            'TotaalWegvoertuigen_1',
            'TrekkerVoorOplegger_13',
            'VrachtautoExclTrekkerVoorOplegger_12']

In [18]:
dfprovince = dfposts[dfposts["RegioS"].str.startswith("PV")]
print(dfprovince)

     Aanhangwagen_17  Bestelauto_11  Bus_15   ID  \
35             44354          31359      99   35   
36             45012          32179     291   36   
37             45677          33225     343   37   
38             46662          34112     346   38   
39             47653          34680     372   39   
..               ...            ...     ...  ...   
114            70317          52209     351  114   
115            72178          53705     332  115   
116            74049          54396     361  116   
117            74725          55128     362  117   
118            75962          56734     352  118   

     InBezitNatuurlijkePersonenRelatief_6  InBezitNatuurlijkePersonen_5  \
35                                    431                        251789   
36                                    436                        255316   
37                                    444                        260707   
38                                    450                        265561   


In [19]:
columns_vehicledb = ['Aanhangwagen_17',
            'Bestelauto_11',
            'Bus_15',
            'InBezitNatuurlijkePersonenRelatief_6',
            'InBezitNatuurlijkePersonen_5',
            'MotorfietsenRelatief_20',
            'Oplegger_18',
            'PersonenautoSRelatief_4',
            'SpeciaalVoertuig_14',
            'TotaalAanhangwagensEnOpleggers_16',
            'TotaalAanhangwagensEnOpleggers_9',
            'TotaalBedrijfsmotorvoertuigen_10',
            'TotaalBedrijfsmotorvoertuigen_8',
            'TotaalBedrijfsvoertuigen_7',
            'TotaalMotorfietsen_19',
            'TotaalMotorvoertuigen_2',
            'TotaalPersonenautoS_3',
            'TotaalWegvoertuigen_1',
            'TrekkerVoorOplegger_13',
            'VrachtautoExclTrekkerVoorOplegger_12']
# encode the data in order to link them to the right province (for visualization)
pvencoding = {'PV20':'Groningen',
              'PV21':'Friesland',
              'PV22':'Drenthe',
              'PV23':'Overijssel',
              'PV24':'Flevoland',
              'PV25':'Gelderland',
              'PV26':'Utrecht',
              'PV27':'Noord-Holland',
              'PV28':'Zuid-Holland',
              'PV29':'Zeeland',
              'PV30':'Noord-Brabant',
              'PV31':'Limburg'}


dfprovince = dfprovince.groupby('RegioS')[columns_vehicledb].sum().reset_index() # group the regios together and calculate the sum from columns_vehicledb per regio
dfprovince["Sum"] = dfprovince[columns_vehicledb].sum(axis=1) #calculate for each row the sum
dfprovince['RegioS'] = dfprovince['RegioS'].astype(str).str.strip()
dfprovince['RegioS'] = dfprovince['RegioS'].map(pvencoding) #transform the initial station_numbers using the pvencoding by implementing the map funtion
df_vehicles = dfprovince[["RegioS","Sum"]].copy()
print(df_vehicles)

           RegioS       Sum
0       Groningen  11936457
1       Friesland  14990976
2         Drenthe  11755668
3      Overijssel  25136738
4       Flevoland  12550586
5      Gelderland  43979534
6         Utrecht  27202822
7   Noord-Holland  47438165
8    Zuid-Holland  62190626
9         Zeeland   8732228
10  Noord-Brabant  57201144
11        Limburg  23560147


In [20]:
print(df_vehicles['Sum'].skew())

0.7002018218075269


In [21]:
df_vehicles['Sum_log'] = np.log(df_vehicles['Sum'])
print(df_vehicles['Sum_log'])
print(df_vehicles['Sum_log'].skew())

0     16.295108
1     16.522959
2     16.279846
3     17.039841
4     16.345278
5     17.599235
6     17.118831
7     17.674938
8     17.945715
9     15.982531
10    17.862084
11    16.975067
Name: Sum_log, dtype: float64
0.10440535907135731


In [22]:
# Standardize the values for number of vehicles
standardization = df_vehicles[['Sum_log']]

standardize = StandardScaler().fit_transform(standardization)

df_vehicles_standardized = df_vehicles.copy()
df_vehicles_standardized['Sum_st'] = pd.DataFrame(standardize, columns=standardization.columns)

print(df_vehicles_standardized)

           RegioS       Sum    Sum_log    Sum_st
0       Groningen  11936457  16.295108 -1.028907
1       Friesland  14990976  16.522959 -0.681598
2         Drenthe  11755668  16.279846 -1.052171
3      Overijssel  25136738  17.039841  0.106275
4       Flevoland  12550586  16.345278 -0.952434
5      Gelderland  43979534  17.599235  0.958949
6         Utrecht  27202822  17.118831  0.226679
7   Noord-Holland  47438165  17.674938  1.074341
8    Zuid-Holland  62190626  17.945715  1.487082
9         Zeeland   8732228  15.982531 -1.505362
10  Noord-Brabant  57201144  17.862084  1.359605
11        Limburg  23560147  16.975067  0.007542


In [23]:
# Assign the vehicles values to the df
def get_vehicles(row):
    province = row['Province']
    match = df_vehicles_standardized[df_vehicles_standardized['RegioS'] == province]
    return match['Sum_st'].iloc[0] if not match.empty else 0

if 'vehicles_tot' not in df_values.columns:
    df_values['vehicles_tot'] = 0

df_values['vehicles_tot'] = df_values.apply(get_vehicles, axis=1)


In [24]:
display(df_values)

,Year_Month,Province,Average NO2 Value,Average PM2.5 Value,Average PM10 Value,area_km2_st,vehicles_tot
0,1990-01-01,Drenthe,1.324917,NaN,NaN,0.000000,-1.052171
1,1990-01-01,Flevoland,1.764091,NaN,NaN,0.000000,-0.952434
2,1990-01-01,Friesland,1.215124,NaN,NaN,0.000000,-0.681598
3,1990-01-01,Gelderland,2.422852,NaN,NaN,0.000000,0.958949
4,1990-01-01,Groningen,1.215124,NaN,NaN,0.000000,-1.028907
...,...,...,...,...,...,...,...
1435,2024-12-01,Noord-Holland,0.336776,0.122001,-0.208369,2.276287,1.074341
1436,2024-12-01,Overijssel,-0.870952,-0.956227,-1.362410,0.000000,0.106275
1437,2024-12-01,Utrecht,-0.102398,0.391558,-0.669985,0.137367,0.226679
1438,2024-12-01,Zeeland,-0.761158,NaN,-0.208369,0.000000,-1.505362


In [25]:
# Save the df as a csv file
df_values.to_csv('df_values.csv')